# Spectral Waterfall Minimal

Compact notebook front end for the production spectral-waterfall workflow.

Open it with the same Python environment as `run_spectral_waterfall.py`.

Use it when you want to:
- point the workflow at a different config or run
- narrow the figure to selected experiments, stations, or size ranges
- switch to LV1B plume-path NetCDFs from `run_lv1_tracking.py`
- explore the spectral waterfall **interactively** with a time (`itime`) slider (no PNG export)

Optional batch export (PNG frames + MP4) still lives on `SpectralWaterfallNotebook.render` if you call it from code; this notebook focuses on the slider workflow.

**Footer ice ⟨D⟩:** very small ridge-integrated ice can inflate the mean diameter. `growth_ice_sum_floor_q` and `growth_ice_mask_until_min` in `psd_process_evolution.yaml` hide those early-time artifacts.

In [1]:
from __future__ import annotations

import argparse
import csv
import re
import sys
import time
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import Video, display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repo root so imports and output paths stay stable from any notebook cwd."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "src/utilities/spectral_waterfall.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook inside the polarcap_analysis repository.")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
for path in (REPO_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from utilities.process_budget_data import build_process_budget_cfg_from_dataset, load_process_budget_data
from utilities.process_rates import first_plume_ridge_anchor
from utilities.processing_paths import get_output_root
from utilities.style_profiles import apply_publication_style
from utilities.spectral_waterfall import (
    _MinMaxAction,
    _build_mp4,
    _build_time_window,
    _ffmpeg_path,
    _group_existing_frames,
    _precompute_growth_overlays,
    _read_yaml,
    _remove_existing_frames,
    _save_frame,
    _sorted_frame_paths,
    _station_tag,
    _waterfall_cfg,
    plot_spectral_waterfall,
)
from scripts.processing_chain.run_publication_figures import _load_publication_config

apply_publication_style()
print(f"repo: {REPO_ROOT}")

repo: /Users/schimmel/Library/Mobile Documents/com~apple~CloudDocs/code/polarcap/python/polarcap_analysis


In [2]:
@dataclass
class SpectralWaterfallSettings:
    '''Small notebook control surface; keep stable defaults in YAML.'''

    kind: str = 'Q'  # Choose the displayed unit family: mass (`Q`) or number (`N`).
    config_path: Path = REPO_ROOT / 'config/psd_process_evolution.yaml'  # PSD + process-evolution defaults (shared process-budget YAML).
    publication_config_path: Path = REPO_ROOT / 'config/publication_figures.yaml'  # Reuse publication presets.
    use_publication_defaults: bool = False  # Start from paper-style limits and MP4 defaults.
    exp_ids: list[int] | None = None  # Restrict the run to selected experiment ids.
    range_keys: list[str] | None = None  # Restrict the figure to selected size families.
    station_ids: list[int] | None = None  # Restrict the figure to selected station columns.
    workers: int | None = None  # Trade render speed against memory use and debugging simplicity.
    input_mode: Literal['process_budget', 'lv1b_plume'] = 'process_budget'  # Keep the classic meteogram path unless LV1B plume files are requested.
    lv1b_processed_root: Path | None = None  # Optional processed-root override; accepts processed, cs_run, or lv1_paths directories.
    lv1b_kinds: tuple[str, ...] = ('extreme', 'integrated')  # Add `vertical` for path × altitude × spectral columns from LV1B extraction.
    lv1b_tracer: str = 'nf'  # Tracking tracer used in LV1B file names.
    lv1b_multi_cell_mode: Literal['sum', 'mean', 'max', 'combine'] = 'sum'  # How to reduce nearby tracked cells before plotting (`combine` collapses all cells per kind).
    lv1b_vertical_ice_reduce: Literal['mean', 'max', 'sum', 'p25', 'p50', 'p75', 'p99'] = 'mean'  # For `vertical` kind only: collapse altitude with this statistic (column ice / mass / spectral fields).
    sw_overrides: dict[str, Any] = field(default_factory=dict)  # Override a few waterfall options without editing YAML.


class SpectralWaterfallNotebook:
    '''Notebook runner that keeps the production render path but trims the interface to a few high-impact switches.'''

    def __init__(self, settings: SpectralWaterfallSettings):
        self.settings = settings
        self.input_mode = settings.input_mode
        overrides = dict(settings.sw_overrides)
        self._sw_override_keys = set(overrides.keys())
        kind_override = overrides.pop('kind', None)
        effective_kind = str(kind_override if kind_override is not None else settings.kind).strip().upper()

        self.cfg_yaml = _read_yaml(settings.config_path)
        self.sw_cfg = _waterfall_cfg(self.cfg_yaml, kind_hint=effective_kind)
        self.sw_cfg['kind'] = effective_kind
        self.default_mp4 = False
        if settings.use_publication_defaults:
            self.default_mp4 = self._apply_publication_defaults()
        # Publication presets may carry a --kind; keep the notebook setting as the source of truth.
        self.sw_cfg['kind'] = effective_kind
        self.sw_cfg.update(overrides)

        self.kind = self.sw_cfg['kind']
        self.kind_label = 'number' if self.kind == 'N' else 'mass'
        self.kind_dir = 'N' if self.kind == 'N' else 'M'
        self.config_cs_run = self.cfg_yaml.get('ensemble', {}).get('cs_run', 'unknown_cs_run')
        self.cs_run = self.config_cs_run
        self.cs_run_tag = self.cs_run.replace('/', '_')
        self.lv1b_paths_dir: Path | None = None
        self.lv1b_expnames: list[str] = []

        selection = self.cfg_yaml.get('selection', {})
        plotting = self.cfg_yaml.get('plotting', {})
        render = plotting.get('render', {})

        self.mp4_root = REPO_ROOT / 'output/gfx/mp4'
        self.frame_dpi = int(render.get('frame_dpi', 300))
        self.frame_png_compress = int(render.get('frame_png_compress', 1))
        self.mp4_fps = int(render.get('mp4_fps', 5))
        default_workers = settings.workers if settings.workers is not None else render.get('workers', 4)
        self.workers = max(1, int(default_workers))
        self.ffmpeg_cmd = _ffmpeg_path()
        self._refresh_output_roots()

        if self.input_mode == 'lv1b_plume':
            self._load_lv1b_context(selection=selection, plotting=plotting)
            self.ridge_context = {}
            self.growth_overlays = {}
            self.growth_csv_rows = []
            self.growth_series = {}
            self.ridge_ref_height_m = {}
            self.growth_csv_path = None
            return

        self.cfg = load_process_budget_data(REPO_ROOT, config_path=settings.config_path)
        self.station_ids = settings.station_ids or selection.get('plot_station_ids', self.cfg['plot_stn_ids'])
        self.plot_exp_ids = settings.exp_ids or selection.get('plot_experiment_ids', self.cfg['plot_exp_ids'])
        self.plot_range_keys = settings.range_keys or plotting.get('plot_range_keys', self.cfg['plot_range_keys'])
        self.height_sel_m = plotting.get('height_sel_m', self.cfg['height_sel_m'])
        self.time_window = _build_time_window(self.cfg_yaml, self.cfg)
        self.stn_tag = _station_tag(self.station_ids)

        self.ridge_context = self._build_ridge_context()
        (
            self.growth_overlays,
            self.growth_csv_rows,
            self.growth_series,
            self.ridge_ref_height_m,
        ) = _precompute_growth_overlays(
            plot_exp_ids=self.plot_exp_ids,
            plot_range_keys=self.plot_range_keys,
            station_ids=self.station_ids,
            time_window=self.time_window,
            cfg=self.cfg,
            kind=self.kind,
            sw_cfg=self.sw_cfg,
            ridge_context=self.ridge_context,
            ds=self.cfg.get("ds"),
        )
        self.growth_csv_path = self._write_growth_csv()

    def _refresh_output_roots(self) -> None:
        self.cs_run_tag = self.cs_run.replace('/', '_')
        self.frame_root = REPO_ROOT / 'output/gfx/png/05' / self.cs_run
        self.csv_root = REPO_ROOT / 'output/gfx/csv/05' / self.cs_run_tag

    def _apply_publication_defaults(self) -> bool:
        publication_cfg = _load_publication_config(self.settings.publication_config_path)
        lookup_key = f'spectral_waterfall_{self.sw_cfg["kind"]}'
        tokens = publication_cfg.get(lookup_key, [])
        if not tokens:
            return False

        parser = argparse.ArgumentParser(add_help=False)
        parser.add_argument('--kind', choices=['N', 'Q', 'n', 'q'], default=None)
        parser.add_argument('--normalize-mode', choices=['none', 'bin', 'panel'], default=None)
        parser.add_argument('--plot-style', choices=['bars', 'lines', 'steps'], default=None)
        parser.add_argument('--yscale', choices=['symlog', 'linear', 'log'], default=None)
        parser.add_argument('--linthresh-w', type=float, default=None)
        parser.add_argument('--linthresh-f', type=float, default=None)
        parser.add_argument('--linscale', type=float, default=None)
        parser.add_argument('--xlim-w', action=_MinMaxAction, default=None)
        parser.add_argument('--xlim-f', action=_MinMaxAction, default=None)
        parser.add_argument('--ylim-w', action=_MinMaxAction, default=None)
        parser.add_argument('--ylim-f', action=_MinMaxAction, default=None)
        parser.add_argument('--psd-ylim-w', action=_MinMaxAction, default=None)
        parser.add_argument('--psd-ylim-f', action=_MinMaxAction, default=None)
        parser.add_argument('--workers', type=int, default=None)
        parser.add_argument('--mp4', action='store_true')
        parser.add_argument('--mp4-only', action='store_true')
        parser.add_argument('--no-psd-twin', action='store_true')
        ns = parser.parse_args(tokens)

        if ns.kind is not None:
            self.sw_cfg['kind'] = ns.kind.upper()

        scalar_map = {
            'normalize_mode': 'normalize_mode',
            'plot_style': 'plot_style',
            'yscale': 'yscale',
            'linthresh_w': 'linthresh_W',
            'linthresh_f': 'linthresh_F',
            'linscale': 'linscale',
        }
        range_map = {
            'xlim_w': 'xlim_W',
            'xlim_f': 'xlim_F',
            'ylim_w': 'ylim_W',
            'ylim_f': 'ylim_F',
            'psd_ylim_w': 'psd_ylim_W',
            'psd_ylim_f': 'psd_ylim_F',
        }
        for attr, key in scalar_map.items():
            value = getattr(ns, attr)
            if value is not None:
                self.sw_cfg[key] = value
        for attr, key in range_map.items():
            value = getattr(ns, attr)
            if value is not None:
                self.sw_cfg[key] = value[0]
        if ns.no_psd_twin:
            self.sw_cfg['show_psd_twin'] = False
        if ns.workers is not None and self.settings.workers is None:
            self.settings.workers = ns.workers
        return bool(ns.mp4 or ns.mp4_only)

    def _resolve_lv1b_paths_dir(self) -> Path:
        candidates: list[Path] = []
        configured = self.settings.lv1b_processed_root
        if configured is not None:
            base = Path(configured).expanduser()
            if base.name == 'lv1_paths':
                candidates.append(base)
            else:
                candidates.extend([base / self.cs_run / 'lv1_paths', base / 'lv1_paths', base])

        auto_root = Path(get_output_root(cs_run=self.cs_run)).expanduser()
        candidates.extend([
            auto_root / self.cs_run / 'lv1_paths',
            auto_root / 'lv1_paths',
            REPO_ROOT / 'data' / 'processed' / self.cs_run / 'lv1_paths',
            REPO_ROOT / 'scripts' / 'data' / 'registry' / 'processed' / self.cs_run / 'lv1_paths',
        ])

        unique: list[Path] = []
        seen: set[str] = set()
        for candidate in candidates:
            resolved = candidate.resolve()
            key = str(resolved)
            if key in seen:
                continue
            seen.add(key)
            unique.append(resolved)

        for path in unique:
            if path.is_dir():
                return path

        tried = '\n'.join(f'- {path}' for path in unique)
        raise FileNotFoundError(
            f'Could not locate LV1B plume-path files for {self.cs_run}. Tried:\n{tried}'
        )

    @staticmethod
    def _cell_index(path: Path) -> int:
        match = re.search(r'_cell(\d+)\.nc$', path.name)
        return int(match.group(1)) if match else -1

    @staticmethod
    def _preprocess_lv1b_ds(ds: xr.Dataset) -> xr.Dataset:
        if 'path' in ds.dims and 'time' in ds.coords and 'time' not in ds.dims:
            ds = ds.swap_dims({'path': 'time'})
        if 'time' not in ds.dims:
            raise ValueError('LV1B dataset does not expose a time dimension after path->time conversion.')
        if not np.issubdtype(ds['time'].dtype, np.datetime64):
            try:
                ds = xr.decode_cf(ds)
            except Exception:
                pass
        if np.issubdtype(ds['time'].dtype, np.datetime64):
            ds = ds.assign_coords(time=np.asarray(ds['time'].values).astype('datetime64[ns]'))
        if 'time' in ds.indexes:
            time_index = ds.indexes['time']
            if time_index.has_duplicates:
                ds = ds.isel(time=~time_index.duplicated())
        return ds.sortby('time')

    @staticmethod
    def _diameter_to_um(values: np.ndarray, units: str | None) -> np.ndarray:
        units_norm = (units or '').lower()
        arr = np.asarray(values, dtype=float)
        if 'mm' in units_norm:
            return arr * 1.0e3
        if 'um' in units_norm or 'mum' in units_norm or 'mic' in units_norm or arr.max(initial=0.0) > 1.0e-2:
            return arr
        return arr * 1.0e6

    @staticmethod
    def _frame_edges_from_times(frame_times: np.ndarray) -> list[np.datetime64]:
        times = np.asarray(frame_times).astype('datetime64[ns]')
        if times.size == 0:
            raise ValueError('LV1B mode needs at least one time step.')
        if times.size == 1:
            step = np.timedelta64(10, 's')
        else:
            diffs = np.diff(times)
            diffs = diffs[diffs > np.timedelta64(0, 'ns')]
            step = diffs.min() if diffs.size else np.timedelta64(10, 's')
        return [*(np.datetime64(t, 'ns') for t in times), np.datetime64(times[-1] + step, 'ns')]

    @staticmethod
    def _lv1b_sum_var_name(name: str) -> str | None:
        match = re.fullmatch(r'd(?P<base>[a-z0-9_]+)_sum', str(name).lower())
        if match is None:
            return None
        return f"SUM_{match.group('base').upper()}"

    def _open_lv1b_dataset(self, files: list[Path]) -> xr.Dataset:
        raw: list[xr.Dataset] = []
        time_blocks: list[np.ndarray] = []
        for file_path in sorted(files, key=self._cell_index):
            with xr.open_dataset(file_path) as opened:
                ds = self._preprocess_lv1b_ds(opened.load())
            cell_id = self._cell_index(file_path)
            raw.append(ds.expand_dims(cell=[cell_id if cell_id >= 0 else len(raw)]))
            time_blocks.append(np.asarray(ds['time'].values).astype('datetime64[ns]'))

        if not raw:
            raise ValueError('LV1B mode needs at least one plume-path file.')
        target_time = np.unique(np.concatenate(time_blocks))
        aligned = [
            ds.reindex(time=target_time, method='nearest', tolerance=np.timedelta64(5, 's'), fill_value=np.nan)
            for ds in raw
        ]
        return xr.concat(aligned, dim='cell')

    @staticmethod
    def _lv1b_latlon_names(ds_kind: xr.Dataset) -> tuple[str, str] | None:
        candidates = [('latitude', 'longitude'), ('plume_latitude', 'plume_longitude')]
        for lat_name, lon_name in candidates:
            if lat_name in ds_kind or lat_name in ds_kind.coords:
                if lon_name in ds_kind or lon_name in ds_kind.coords:
                    return lat_name, lon_name
        return None

    @staticmethod
    def _lv1b_cell_ids(ds_kind: xr.Dataset) -> list[int]:
        if 'cell' not in ds_kind.dims:
            return []
        return [int(value) for value in np.asarray(ds_kind['cell'].values)]

    @staticmethod
    def _lv1b_vertical_dim_name(ds_kind: xr.Dataset) -> str | None:
        for name in ('altitude', 'level', 'height'):
            if name in ds_kind.dims:
                return name
        return None

    def _lv1b_vertical_reduce_mode(self) -> str:
        mode = str(self.settings.lv1b_vertical_ice_reduce).strip().lower()
        allowed = {'mean', 'max', 'sum', 'p25', 'p50', 'p75', 'p99'}
        if mode not in allowed:
            raise ValueError(
                f'Unsupported lv1b_vertical_ice_reduce {mode!r}. Use mean, max, sum, p25, p50, p75, or p99.'
            )
        return mode

    def _lv1b_reduce_along_vertical(self, da: xr.DataArray, vdim: str, how: str, *, coord_policy_mean: bool) -> xr.DataArray:
        if vdim not in da.dims:
            return da
        eff = 'mean' if coord_policy_mean else how
        if eff == 'mean':
            return da.mean(vdim, skipna=True)
        if eff == 'max':
            return da.max(vdim, skipna=True)
        if eff == 'sum':
            return da.sum(vdim, skipna=True, min_count=1)
        qmap = {'p25': 0.25, 'p50': 0.5, 'p75': 0.75, 'p99': 0.99}
        out = da.quantile(qmap[eff], dim=vdim, skipna=True)
        if 'quantile' in out.dims:
            out = out.squeeze('quantile', drop=True)
        return out

    def _lv1b_collapse_vertical(self, ds_kind: xr.Dataset, *, vdim: str, how: str) -> xr.Dataset:
        lonlat_mean_only = {'latitude', 'longitude', 'latitude2d', 'longitude2d'}
        reduced_vars: dict[str, xr.DataArray] = {}
        for name, da in ds_kind.data_vars.items():
            if vdim not in da.dims:
                reduced_vars[name] = da
            else:
                reduced_vars[name] = self._lv1b_reduce_along_vertical(
                    da, vdim, how, coord_policy_mean=name in lonlat_mean_only
                )
        out = xr.Dataset(reduced_vars, attrs=dict(ds_kind.attrs))
        for cname, cda in ds_kind.coords.items():
            if cname in out.coords or cname == vdim:
                continue
            if vdim not in cda.dims:
                out = out.assign_coords({cname: cda})
            elif cname in lonlat_mean_only:
                out = out.assign_coords({cname: cda.mean(vdim, skipna=True)})
            else:
                out = out.assign_coords({
                    cname: self._lv1b_reduce_along_vertical(cda, vdim, how, coord_policy_mean=False)
                })
        out.attrs['lv1b_vertical_reduce'] = how
        out.attrs['lv1b_vertical_dim'] = vdim
        return out

    def _lv1b_multi_cell_mode(self) -> str:
        mode = str(self.settings.lv1b_multi_cell_mode).strip().lower()
        if mode not in {'sum', 'mean', 'max', 'combine'}:
            raise ValueError(
                f'Unsupported lv1b_multi_cell_mode {mode!r}. Use sum, mean, max, or combine.'
            )
        return mode

    def _lv1b_dominant_cell(self, ds_kind: xr.Dataset, cell_group: list[int]) -> int:
        if not cell_group:
            raise ValueError('LV1B dominant-cell lookup needs at least one cell.')
        if len(cell_group) == 1 or 'cell' not in ds_kind.dims:
            return int(cell_group[0])

        ds_sel = ds_kind.sel(cell=cell_group)
        tracer_candidates = [self.settings.lv1b_tracer, 'nf', 'qf', 'nw', 'qw']
        score_da = None
        for name in tracer_candidates:
            if name in ds_sel.data_vars:
                score_da = ds_sel[name]
                break
        if score_da is None:
            for name in ds_sel.data_vars:
                if 'cell' in ds_sel[name].dims:
                    score_da = ds_sel[name]
                    break
        if score_da is None:
            return int(cell_group[0])

        reduce_dims = [dim for dim in score_da.dims if dim != 'cell']
        scores = score_da.fillna(0).sum(dim=reduce_dims, skipna=True) if reduce_dims else score_da
        values = np.asarray(scores.values, dtype=float)
        if values.ndim == 0 or not np.isfinite(values).any():
            return int(cell_group[0])
        best_idx = int(np.nanargmax(values))
        return int(np.asarray(scores['cell'].values)[best_idx])

    def _lv1b_group_label(self, plume_kind: str, cell_group: list[int]) -> str:
        vtag = f' ({self.settings.lv1b_vertical_ice_reduce})' if plume_kind == 'vertical' else ''
        if len(cell_group) == 1:
            return f'{plume_kind.title()} cell {cell_group[0]}{vtag}'

        mode = self._lv1b_multi_cell_mode()
        if mode == 'combine':
            return f'{plume_kind.title()} combined ({len(cell_group)} cells){vtag}'

        cells = '+'.join(str(cell) for cell in cell_group)
        if mode == 'mean':
            return f'{plume_kind.title()} mean cells {cells}{vtag}'
        if mode == 'max':
            return f'{plume_kind.title()} max cell from {cells}{vtag}'
        return f'{plume_kind.title()} sum cells {cells}{vtag}'

    @staticmethod
    def _lv1b_complete_link_distance(
        cluster_a: list[int],
        cluster_b: list[int],
        pair_distance_m: dict[tuple[int, int], float],
    ) -> float:
        distances: list[float] = []
        for cell_a in cluster_a:
            for cell_b in cluster_b:
                key = tuple(sorted((int(cell_a), int(cell_b))))
                if cell_a == cell_b:
                    continue
                if key not in pair_distance_m:
                    return float('inf')
                distances.append(pair_distance_m[key])
        return max(distances) if distances else float('inf')

    def _lv1b_merge_distance_m(self, ds_kind: xr.Dataset) -> float:
        dx = float(ds_kind.attrs.get('delta_x', np.nan))
        dy = float(ds_kind.attrs.get('delta_y', np.nan))
        spacing = max(dx, dy) if np.isfinite(dx) or np.isfinite(dy) else np.nan
        if np.isfinite(spacing) and spacing > 0.0:
            return 1.5 * spacing
        return 1000.0

    def _lv1b_pair_distance_m(self, ds_kind: xr.Dataset, cell_a: int, cell_b: int) -> float:
        latlon_names = self._lv1b_latlon_names(ds_kind)
        if latlon_names is None:
            return float('inf')
        lat_name, lon_name = latlon_names
        lat_a = np.asarray(ds_kind[lat_name].sel(cell=cell_a).values, dtype=float)
        lat_b = np.asarray(ds_kind[lat_name].sel(cell=cell_b).values, dtype=float)
        lon_a = np.asarray(ds_kind[lon_name].sel(cell=cell_a).values, dtype=float)
        lon_b = np.asarray(ds_kind[lon_name].sel(cell=cell_b).values, dtype=float)

        track_a = np.column_stack([lat_a[np.isfinite(lat_a) & np.isfinite(lon_a)], lon_a[np.isfinite(lat_a) & np.isfinite(lon_a)]])
        track_b = np.column_stack([lat_b[np.isfinite(lat_b) & np.isfinite(lon_b)], lon_b[np.isfinite(lat_b) & np.isfinite(lon_b)]])
        if track_a.size == 0 or track_b.size == 0:
            return float('inf')

        def _one_way(source: np.ndarray, target: np.ndarray) -> float:
            distances: list[float] = []
            for lat_pt, lon_pt in source:
                mean_lat = 0.5 * (lat_pt + target[:, 0])
                dx_m = (lon_pt - target[:, 1]) * 111132.0 * np.cos(np.deg2rad(mean_lat))
                dy_m = (lat_pt - target[:, 0]) * 111132.0
                distances.append(float(np.nanmin(np.sqrt(dx_m**2 + dy_m**2))))
            return float(np.nanmean(distances)) if distances else float('inf')

        return 0.5 * (_one_way(track_a, track_b) + _one_way(track_b, track_a))

    def _lv1b_cell_groups(
        self,
        ds_kind: xr.Dataset,
        *,
        fallback_groups: list[list[int]] | None = None,
    ) -> list[list[int]]:
        cell_ids = self._lv1b_cell_ids(ds_kind)
        if len(cell_ids) <= 1:
            return [cell_ids] if cell_ids else []
        latlon_names = self._lv1b_latlon_names(ds_kind)
        if latlon_names is None:
            if fallback_groups:
                grouped = [[cell for cell in group if cell in cell_ids] for group in fallback_groups]
                groups = [group for group in grouped if group]
                assigned = {cell for group in groups for cell in group}
                groups.extend([[cell] for cell in cell_ids if cell not in assigned])
                return groups
            return [[cell] for cell in cell_ids]

        threshold_m = self._lv1b_merge_distance_m(ds_kind)
        pair_distance_m: dict[tuple[int, int], float] = {}
        for idx, cell_a in enumerate(cell_ids):
            for cell_b in cell_ids[idx + 1 :]:
                pair_distance_m[(cell_a, cell_b)] = self._lv1b_pair_distance_m(ds_kind, cell_a, cell_b)

        clusters = [[cell] for cell in cell_ids]
        while True:
            best_pair: tuple[int, int] | None = None
            best_distance = float('inf')
            for idx, cluster_a in enumerate(clusters):
                for jdx in range(idx + 1, len(clusters)):
                    dist = self._lv1b_complete_link_distance(cluster_a, clusters[jdx], pair_distance_m)
                    if dist < best_distance:
                        best_distance = dist
                        best_pair = (idx, jdx)
            if best_pair is None or not np.isfinite(best_distance) or best_distance > threshold_m:
                break
            left, right = best_pair
            merged = sorted(clusters[left] + clusters[right])
            clusters[left] = merged
            del clusters[right]

        return sorted((sorted(cluster) for cluster in clusters), key=lambda cluster: (cluster[0], len(cluster)))

    def _lv1b_merge_cell_group(self, ds_kind: xr.Dataset, cell_group: list[int]) -> xr.Dataset:
        if not cell_group:
            raise ValueError('LV1B cell group cannot be empty.')
        if 'cell' not in ds_kind.dims:
            return ds_kind

        ds_sel = ds_kind.sel(cell=cell_group)
        mode = self._lv1b_multi_cell_mode()
        reducer_mode = 'sum' if mode == 'combine' else mode
        if len(cell_group) == 1:
            merged = ds_sel.isel(cell=0, drop=True)
            merged.attrs['cell_aggregation_mode'] = mode
            return merged

        if reducer_mode == 'max':
            selected_cell = self._lv1b_dominant_cell(ds_kind, cell_group)
            merged = ds_sel.sel(cell=selected_cell)
            if 'cell' in merged.dims:
                merged = merged.squeeze('cell', drop=True)
            merged = merged.copy()
            merged.attrs.update(ds_sel.attrs)
            merged.attrs['merged_cells'] = ','.join(str(cell) for cell in cell_group)
            merged.attrs['merged_cell_count'] = len(cell_group)
            merged.attrs['cell_aggregation_mode'] = mode
            merged.attrs['selected_cell'] = int(selected_cell)
            return merged

        merged = xr.Dataset(coords={k: v for k, v in ds_sel.coords.items() if 'cell' not in v.dims})
        sum_var_names = {'nw', 'nf', 'qw', 'qf'}
        mean_var_names = {'rho', 'altitude', 'latitude', 'longitude'}
        for raw_name in sorted(ds_sel.data_vars):
            if 'cell' not in ds_sel[raw_name].dims:
                merged[raw_name] = ds_sel[raw_name]
                continue
            if raw_name in mean_var_names:
                merged[raw_name] = ds_sel[raw_name].mean('cell', skipna=True)
                continue
            if reducer_mode == 'sum' and (raw_name in sum_var_names or self._lv1b_sum_var_name(raw_name) is not None):
                merged[raw_name] = ds_sel[raw_name].sum('cell', skipna=True, min_count=1)
                continue
            merged[raw_name] = ds_sel[raw_name].mean('cell', skipna=True)

        for coord_name in ('altitude', 'latitude', 'longitude'):
            if coord_name in ds_sel.coords and 'cell' in ds_sel[coord_name].dims:
                merged[coord_name] = ds_sel[coord_name].mean('cell', skipna=True)

        merged.attrs.update(ds_sel.attrs)
        merged.attrs['merged_cells'] = ','.join(str(cell) for cell in cell_group)
        merged.attrs['merged_cell_count'] = len(cell_group)
        merged.attrs['cell_aggregation_mode'] = mode
        return merged

    def _lv1b_station_dataset(self, ds_mean: xr.Dataset, station_idx: int) -> xr.Dataset:
        time_values = np.asarray(ds_mean['time'].values).astype('datetime64[ns]')
        diameter_um = self._diameter_to_um(
            np.asarray(ds_mean['diameter'].values, dtype=float),
            ds_mean['diameter'].attrs.get('units'),
        )
        bins = np.arange(diameter_um.size, dtype=int)
        station_ds = xr.Dataset(
            coords={
                'time': time_values,
                'height_level': [0.5],
                'station': [station_idx],
                'bins': bins,
                'diameter_um': ('bins', diameter_um),
            }
        )

        rho = ds_mean['rho'] if 'rho' in ds_mean.data_vars else xr.DataArray(
            np.ones(time_values.size, dtype=float),
            dims=('time',),
            coords={'time': time_values},
        )
        rho = rho.astype(float)
        rho_safe = xr.where(np.abs(rho) > 0.0, rho, np.nan)

        def _expand_profile(da: xr.DataArray) -> xr.DataArray:
            base = xr.DataArray(
                np.asarray(da.values, dtype=float),
                dims=('time',),
                coords={'time': time_values},
            )
            out = base.expand_dims(height_level=[0.5], station=[station_idx])
            return out.transpose('time', 'height_level', 'station')

        def _expand_spectral(da: xr.DataArray) -> xr.DataArray:
            base = xr.DataArray(
                np.asarray(da.values, dtype=float),
                dims=('time', 'bins'),
                coords={'time': time_values, 'bins': bins},
            )
            out = base.expand_dims(height_level=[0.5], station=[station_idx])
            return out.transpose('time', 'height_level', 'station', 'bins')

        station_ds['RHO'] = _expand_profile(rho)
        if 'nw' in ds_mean.data_vars:
            station_ds['NW'] = _expand_spectral(ds_mean['nw'] * 1.0e6 / rho_safe)
        if 'nf' in ds_mean.data_vars:
            station_ds['NF'] = _expand_spectral(ds_mean['nf'] * 1.0e3 / rho_safe)
        if 'qw' in ds_mean.data_vars:
            station_ds['QW'] = _expand_spectral(ds_mean['qw'] / (rho_safe * 1.0e3))
        if 'qf' in ds_mean.data_vars:
            station_ds['QF'] = _expand_spectral(ds_mean['qf'] / rho_safe)

        for aux_name in ('altitude', 'latitude', 'longitude'):
            if aux_name in ds_mean.data_vars:
                station_ds[f'plume_{aux_name}'] = _expand_profile(ds_mean[aux_name])
            elif aux_name in ds_mean.coords and aux_name != 'time':
                station_ds[f'plume_{aux_name}'] = _expand_profile(ds_mean[aux_name])

        for raw_name in sorted(ds_mean.data_vars):
            sum_name = self._lv1b_sum_var_name(raw_name)
            if sum_name is None:
                continue
            station_ds[sum_name] = _expand_spectral(ds_mean[raw_name])

        return station_ds

    def _load_lv1b_context(self, *, selection: dict[str, Any], plotting: dict[str, Any]) -> None:
        allowed_kinds = {'extreme', 'integrated', 'vertical'}
        requested_kinds: list[str] = []
        for raw_kind in self.settings.lv1b_kinds:
            plume_kind = str(raw_kind).strip().lower()
            if not plume_kind:
                continue
            if plume_kind not in allowed_kinds:
                raise ValueError(
                    f'Unsupported LV1B kind {raw_kind!r}. Use extreme, integrated, and/or vertical.'
                )
            if plume_kind not in requested_kinds:
                requested_kinds.append(plume_kind)
        if not requested_kinds:
            raise ValueError('LV1B mode needs at least one plume kind.')

        self.lv1b_paths_dir = self._resolve_lv1b_paths_dir()
        tracer = re.escape(self.settings.lv1b_tracer)
        kind_pattern = '|'.join(re.escape(kind) for kind in requested_kinds)
        pattern = re.compile(
            rf'^data_(?P<cs>cs-.*?__\d{{8}}_\d{{6}})_(?P<exp>\d{{14}})_(?P<kind>{kind_pattern})_plume_path_{tracer}_cell\d+\.nc$'
        )

        discovered: dict[tuple[str, int, str], list[Path]] = {}
        for path in self.lv1b_paths_dir.glob(f'data_*_plume_path_{self.settings.lv1b_tracer}_cell*.nc'):
            match = pattern.match(path.name)
            if match is None:
                continue
            cs_run = match.group('cs')
            exp_id = int(match.group('exp'))
            plume_kind = match.group('kind')
            discovered.setdefault((cs_run, exp_id, plume_kind), []).append(path)

        if not discovered:
            raise FileNotFoundError(
                f'No LV1B plume-path files matched tracer={self.settings.lv1b_tracer!r} under {self.lv1b_paths_dir}.'
            )

        discovered_cs_runs = sorted({cs_run for cs_run, _exp_id, _kind in discovered})
        if self.config_cs_run in discovered_cs_runs:
            self.cs_run = self.config_cs_run
        elif len(discovered_cs_runs) == 1:
            self.cs_run = discovered_cs_runs[0]
        else:
            raise ValueError(
                f'LV1B path {self.lv1b_paths_dir} contains multiple cs_runs {discovered_cs_runs}; '
                'set `config_path` or `lv1b_processed_root` to target one run.'
            )
        self._refresh_output_roots()

        discovered_exp_ids = sorted({exp_id for cs_run, exp_id, _kind in discovered if cs_run == self.cs_run})
        requested_exp_ids = [int(exp_id) for exp_id in self.settings.exp_ids] if self.settings.exp_ids else []
        if not requested_exp_ids:
            selected_actual_exp_ids = discovered_exp_ids
        elif all(exp_id in discovered_exp_ids for exp_id in requested_exp_ids):
            selected_actual_exp_ids = requested_exp_ids
        elif all(0 <= exp_id < len(discovered_exp_ids) for exp_id in requested_exp_ids):
            selected_actual_exp_ids = [discovered_exp_ids[exp_id] for exp_id in requested_exp_ids]
        else:
            raise ValueError(
                f'LV1B experiments {requested_exp_ids} not found for {self.cs_run}. Available flare ids: {discovered_exp_ids}'
            )
        self.lv1b_expnames = [str(exp_id) for exp_id in selected_actual_exp_ids]

        exp_datasets: list[xr.Dataset] = []
        all_times: list[np.ndarray] = []
        station_layout: list[tuple[int, str, list[int]]] | None = None
        for actual_exp_id in selected_actual_exp_ids:
            ds_by_kind: dict[str, xr.Dataset] = {}
            kind_times: list[np.ndarray] = []
            for plume_kind in requested_kinds:
                files = sorted(discovered.get((self.cs_run, actual_exp_id, plume_kind), []), key=self._cell_index)
                if not files:
                    raise FileNotFoundError(
                        f'Missing LV1B files for cs_run={self.cs_run} exp={actual_exp_id} kind={plume_kind} under {self.lv1b_paths_dir}.'
                    )
                ds_kind = self._open_lv1b_dataset(files)
                ds_by_kind[plume_kind] = ds_kind
                kind_times.append(np.asarray(ds_kind['time'].values).astype('datetime64[ns]'))

            target_time = np.unique(np.concatenate(kind_times))
            aligned_by_kind: dict[str, xr.Dataset] = {}
            vreduce = self._lv1b_vertical_reduce_mode()
            for plume_kind, ds_kind in ds_by_kind.items():
                aligned = ds_kind.reindex(
                    time=target_time,
                    method='nearest',
                    tolerance=np.timedelta64(5, 's'),
                    fill_value=np.nan,
                )
                if plume_kind == 'vertical':
                    vdim = self._lv1b_vertical_dim_name(aligned)
                    if vdim is None:
                        raise ValueError(
                            'LV1B vertical plume files must include an altitude/level dimension to collapse.'
                        )
                    aligned = self._lv1b_collapse_vertical(aligned, vdim=vdim, how=vreduce)
                aligned_by_kind[plume_kind] = aligned

            multi_cell_mode = self._lv1b_multi_cell_mode()
            kind_groups: dict[str, list[list[int]]] = {}
            reference_groups: list[list[int]] | None = None
            for plume_kind in requested_kinds:
                if multi_cell_mode == 'combine':
                    kind_groups[plume_kind] = [self._lv1b_cell_ids(aligned_by_kind[plume_kind])]
                    continue
                groups = self._lv1b_cell_groups(aligned_by_kind[plume_kind], fallback_groups=reference_groups)
                kind_groups[plume_kind] = groups
                if reference_groups is None and self._lv1b_latlon_names(aligned_by_kind[plume_kind]) is not None:
                    reference_groups = groups

            current_layout: list[tuple[int, str, list[int]]] = []
            station_counter = 0
            for plume_kind, groups in kind_groups.items():
                for cell_group in groups:
                    current_layout.append((station_counter, plume_kind, cell_group))
                    station_counter += 1
            if station_layout is None:
                station_layout = current_layout
            elif multi_cell_mode == 'combine':
                # In combine-mode, cell ids can differ between experiments; enforce only the kind/station structure.
                left = [(station_idx, plume_kind) for station_idx, plume_kind, _group in station_layout]
                right = [(station_idx, plume_kind) for station_idx, plume_kind, _group in current_layout]
                if left != right:
                    raise ValueError('LV1B combine-mode station layout changed between experiments; use one flare id at a time.')
            elif station_layout != current_layout:
                raise ValueError('LV1B cell clustering changed between experiments; use one flare id at a time.')

            station_datasets: list[xr.Dataset] = []
            for station_idx, plume_kind, cell_group in station_layout:
                if multi_cell_mode == 'combine':
                    cell_group = self._lv1b_cell_ids(aligned_by_kind[plume_kind])
                merged_ds = self._lv1b_merge_cell_group(aligned_by_kind[plume_kind], cell_group)
                station_datasets.append(self._lv1b_station_dataset(merged_ds, station_idx))

            ds_exp = xr.concat(station_datasets, dim='station', fill_value=np.nan)
            exp_datasets.append(ds_exp.expand_dims(expname=[str(actual_exp_id)]))
            all_times.append(target_time)

        if station_layout is None:
            raise ValueError('LV1B mode could not build any station layout.')

        station_labels = {
            station_idx: self._lv1b_group_label(plume_kind, cell_group)
            for station_idx, plume_kind, cell_group in station_layout
        }
        available_station_ids = [station_idx for station_idx, _kind, _group in station_layout]
        selected_station_ids = self.settings.station_ids or available_station_ids
        if any(station_id not in available_station_ids for station_id in selected_station_ids):
            raise ValueError(f'LV1B station_ids must be a subset of {available_station_ids}.')

        ds = xr.concat(exp_datasets, dim='expname', fill_value=np.nan)
        ds = ds.assign_coords(expname=self.lv1b_expnames)

        lv1b_cfg_yaml = deepcopy(self.cfg_yaml)
        lv1b_cfg_yaml.setdefault('ensemble', {})
        lv1b_cfg_yaml['ensemble']['cs_run'] = self.cs_run
        lv1b_cfg_yaml.setdefault('selection', {})
        lv1b_cfg_yaml['selection']['plot_experiment_ids'] = list(range(len(self.lv1b_expnames)))
        lv1b_cfg_yaml['selection']['plot_station_ids'] = selected_station_ids
        lv1b_cfg_yaml['selection']['station_labels'] = station_labels
        lv1b_cfg_yaml.setdefault('plotting', {})
        lv1b_cfg_yaml['plotting']['plot_range_keys'] = self.settings.range_keys or plotting.get('plot_range_keys', ['ALLBB'])
        lv1b_cfg_yaml['plotting']['active_range_key'] = lv1b_cfg_yaml['plotting']['plot_range_keys'][0]
        lv1b_cfg_yaml['plotting']['height_sel_m'] = [0.0, 1.0]

        self.cfg = build_process_budget_cfg_from_dataset(
            ds,
            cfg=lv1b_cfg_yaml,
            repo_root=REPO_ROOT,
            cs_run=self.cs_run,
        )
        forced_defaults = {
            'use_plume_ridge': False,
            'show_growth_footer': False,
            'show_growth_sparkline': False,
            'show_growth_textbox': False,
        }
        override_keys = getattr(self, '_sw_override_keys', set())
        for key, value in forced_defaults.items():
            if key not in override_keys:
                self.sw_cfg[key] = value

        self.station_ids = selected_station_ids
        self.plot_exp_ids = self.cfg['plot_exp_ids']
        self.plot_range_keys = lv1b_cfg_yaml['plotting']['plot_range_keys']
        self.height_sel_m = lv1b_cfg_yaml['plotting']['height_sel_m']
        self.time_window = self._frame_edges_from_times(np.unique(np.concatenate(all_times)))
        self.stn_tag = _station_tag(self.station_ids)

    def _build_ridge_context(self) -> dict[tuple[int, str], dict[int, tuple[int, int, float]]]:
        floor = float(self.sw_cfg.get('ridge_mass_floor', 0.0))
        out: dict[tuple[int, str], dict[int, tuple[int, int, float]]] = {}
        for exp_id in self.plot_exp_ids:
            rates = self.cfg['rates_by_exp'][exp_id]
            ridge_conc = rates.get('spec_conc_Q_F')
            if ridge_conc is None:
                continue
            for range_key in self.plot_range_keys:
                bin_slice = self.cfg['size_ranges'][range_key]['slice']
                per_station: dict[int, tuple[int, int, float]] = {}
                for station_id in self.station_ids:
                    anchor = first_plume_ridge_anchor(ridge_conc, station_id, bin_slice, floor=floor)
                    if anchor is None:
                        continue
                    gti, hi = anchor
                    z_m = float(np.asarray(ridge_conc['height_level'].values)[int(hi)])
                    per_station[station_id] = (gti, hi, z_m)
                out[(exp_id, range_key)] = per_station
        return out

    def _write_growth_csv(self) -> Path | None:
        if not (self.sw_cfg.get('write_growth_csv', False) and self.growth_csv_rows):
            return None
        self.csv_root.mkdir(parents=True, exist_ok=True)
        csv_path = self.csv_root / f'ridge_growth_{self.kind}_{self.stn_tag}.csv'
        with csv_path.open('w', newline='', encoding='utf-8') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(self.growth_csv_rows[0].keys()))
            writer.writeheader()
            writer.writerows(self.growth_csv_rows)
        return csv_path

    def summary(self) -> dict[str, Any]:
        out = {
            'input_mode': self.input_mode,
            'cs_run': self.cs_run,
            'kind': self.kind,
            'stations': self.station_ids,
            'experiments': self.plot_exp_ids,
            'ranges': self.plot_range_keys,
            'n_frames': len(self.time_window) - 1,
            'frame_root': str(self.frame_root),
            'mp4_root': str(self.mp4_root),
            'growth_csv': None if self.growth_csv_path is None else str(self.growth_csv_path),
        }
        if self.input_mode == 'lv1b_plume':
            out['lv1b_paths_dir'] = None if self.lv1b_paths_dir is None else str(self.lv1b_paths_dir)
            out['lv1b_tracer'] = self.settings.lv1b_tracer
            out['lv1b_multi_cell_mode'] = self.settings.lv1b_multi_cell_mode
            out['lv1b_vertical_ice_reduce'] = self.settings.lv1b_vertical_ice_reduce
            out['lv1b_expnames'] = self.lv1b_expnames
        return out

    def interactive_time(
        self,
        exp_id: int | None = None,
        range_key: str | None = None,
        *,
        continuous_slider: bool = False,
    ) -> None:
        '''Redraw the waterfall inside the notebook from an `itime` slider; does not write PNGs.'''
        exp_id = self.plot_exp_ids[0] if exp_id is None else exp_id
        range_key = self.plot_range_keys[0] if range_key is None else range_key
        n = len(self.time_window) - 1
        if n <= 0:
            raise ValueError('time_window yields no frames')

        out = widgets.Output(layout={'border': '1px solid #ddd'})
        slider = widgets.IntSlider(
            value=0, min=0, max=n - 1, step=1, continuous_update=continuous_slider, description='itime',
        )
        _last_fig: list[Any | None] = [None]

        def redraw(itime: int) -> None:
            with out:
                out.clear_output(wait=True)
                if _last_fig[0] is not None:
                    plt.close(_last_fig[0])
                _last_fig[0] = self.preview(itime=int(itime), exp_id=exp_id, range_key=range_key, save=False)
                plt.show()

        def on_change(change: dict[str, Any]) -> None:
            redraw(int(change['new']))

        slider.observe(on_change, names='value')
        display(widgets.VBox([slider, out]))
        redraw(0)

    def _render_figure(self, exp_id: int, range_key: str, itime: int):
        rates = self.cfg['rates_by_exp'][exp_id]
        twindow = slice(self.time_window[itime], self.time_window[itime + 1])

        ctx = self.ridge_context.get((exp_id, range_key), {})
        anchor_map = {station: (gti, hi) for station, (gti, hi, _z) in ctx.items()}
        ref_height = self.ridge_ref_height_m.get((exp_id, range_key), {})
        fallback_height = {station: z_m for station, (_gti, _hi, z_m) in ctx.items()}
        ridge_height = {
            station: (
                float(ref_height[station])
                if station in ref_height and np.isfinite(ref_height[station])
                else fallback_height[station]
            )
            for station in self.station_ids
            if (station in ref_height and np.isfinite(ref_height[station])) or station in fallback_height
        }

        show_psd = bool(self.sw_cfg.get('show_psd_twin'))
        growth_footer = bool(self.sw_cfg.get('show_growth_footer') or self.sw_cfg.get('show_growth_sparkline'))
        fig, _ = plot_spectral_waterfall(
            spec_rates_w=rates[f'spec_rates_{self.kind}_W'],
            spec_rates_f=rates[f'spec_rates_{self.kind}_F'],
            size_ranges=self.cfg['size_ranges'],
            range_key=range_key,
            diameter_um=self.cfg['diameter_um'],
            station_ids=self.station_ids,
            station_labels=self.cfg['station_labels'],
            height_sel_m=self.height_sel_m,
            twindow=twindow,
            unit_label=rates[f'unit_{self.kind}'],
            kind_label=self.kind_label,
            cfg_plot=self.sw_cfg,
            normalize_mode=self.sw_cfg['normalize_mode'],
            plot_style=self.sw_cfg['plot_style'],
            yscale=self.sw_cfg['yscale'],
            spec_conc_w=rates.get(f'spec_conc_{self.kind}_W') if show_psd else None,
            spec_conc_f=rates.get(f'spec_conc_{self.kind}_F') if show_psd else None,
            ridge_conc_f=rates.get('spec_conc_Q_F'),
            conc_unit_label=r'#/m³' if self.kind == 'N' else r'g/m³',
            ridge_anchor_by_station=anchor_map or None,
            ridge_height_m_by_station=ridge_height or None,
            growth_overlay_by_station=self.growth_overlays.get((exp_id, range_key, itime)),
            growth_footer_series_by_station=self.growth_series.get((exp_id, range_key)) if growth_footer else None,
            growth_footer_itime=itime,
            growth_footer_slice_unit='g/m3' if self.kind == 'Q' else '#/m3',
        )
        stem = f'spectral_waterfall_{self.kind}_{self.cs_run_tag}_exp{exp_id}_{self.stn_tag}_{range_key}_itime{itime}'
        out_dir = self.frame_root / f'exp{exp_id}' / self.kind_dir
        return fig, stem, out_dir

    def preview(self, itime: int = 0, exp_id: int | None = None, range_key: str | None = None, save: bool = False):
        exp_id = self.plot_exp_ids[0] if exp_id is None else exp_id
        range_key = self.plot_range_keys[0] if range_key is None else range_key
        fig, stem, out_dir = self._render_figure(exp_id, range_key, itime)
        if save:
            _save_frame(fig, stem, out_dir, self.frame_dpi, self.frame_png_compress)
        return fig

    def _save_render_task(self, task: tuple[int, str, int]) -> str:
        exp_id, range_key, itime = task
        fig, stem, out_dir = self._render_figure(exp_id, range_key, itime)
        _save_frame(fig, stem, out_dir, self.frame_dpi, self.frame_png_compress)
        plt.close(fig)
        return stem

    def build_mp4_only(self) -> list[Path]:
        groups = _group_existing_frames(
            self.frame_root,
            kind=self.kind,
            kind_dir=self.kind_dir,
            cs_run_tag=self.cs_run_tag,
            exp_ids=self.plot_exp_ids,
            range_keys=self.plot_range_keys,
            station_tag=self.stn_tag,
        )
        mp4_paths: list[Path] = []
        for prefix, frames in groups:
            mp4_path = self.mp4_root / f'{prefix}_evolution_nframes{len(frames)}.mp4'
            _build_mp4(self.ffmpeg_cmd, frames, mp4_path, self.mp4_fps)
            mp4_paths.append(mp4_path)
            print(mp4_path)
        return mp4_paths

    def render(self, *, mp4: bool | None = None, mp4_only: bool = False) -> list[Path]:
        mp4 = self.default_mp4 if mp4 is None else mp4
        self.frame_root.mkdir(parents=True, exist_ok=True)
        self.mp4_root.mkdir(parents=True, exist_ok=True)
        if mp4_only:
            return self.build_mp4_only()

        started = time.perf_counter()
        expected_frames = len(self.time_window) - 1
        mp4_paths: list[Path] = []

        for exp_id in self.plot_exp_ids:
            for range_key in self.plot_range_keys:
                frame_dir = self.frame_root / f'exp{exp_id}' / self.kind_dir
                pattern = str(
                    frame_dir
                    / f'spectral_waterfall_{self.kind}_{self.cs_run_tag}_exp{exp_id}_{self.stn_tag}_{range_key}_itime*.png'
                )
                _remove_existing_frames(pattern)
                tasks = [(exp_id, range_key, itime) for itime in range(expected_frames)]

                if self.workers > 1 and len(tasks) > 1:
                    with ThreadPoolExecutor(max_workers=min(self.workers, len(tasks))) as pool:
                        list(pool.map(self._save_render_task, tasks))
                else:
                    for task in tasks:
                        self._save_render_task(task)

                frames = _sorted_frame_paths(pattern, expected_count=expected_frames)
                print(f'exp={exp_id} {range_key}: {len(frames)} frames')
                if mp4 and frames:
                    mp4_path = self.mp4_root / (
                        f'spectral_waterfall_{self.kind}_{self.cs_run_tag}_exp{exp_id}_{self.stn_tag}_{range_key}_evolution_nframes{len(frames)}.mp4'
                    )
                    _build_mp4(self.ffmpeg_cmd, frames, mp4_path, self.mp4_fps)
                    mp4_paths.append(mp4_path)
                    print(mp4_path.resolve().as_uri())

        print(f'done in {time.perf_counter() - started:.1f}s')
        return mp4_paths

    def show_mp4(self, path: Path) -> None:
        display(Video(str(path), embed=True, html_attributes='controls loop'))

In [3]:
# SETTINGS = SpectralWaterfallSettings(
#     kind='Q',  # `Q` plots qf in LV1B mode; switch to `N` for nf.
#     use_publication_defaults=True,  # Pull in the paper-style limits and MP4 preset.
#     exp_ids=[0],  # Process-budget experiment index, or the first discovered LV1B flare id by position.
#     range_keys=['ALLBB'],  # Size family shown by the shared waterfall routine in both modes.
#     station_ids=[0,1,2,3],  # In LV1B mode these select proximity-merged plume-path groups, not fixed kinds.
#     workers=4,  # Speed up a full render; lower this if memory gets tight.
#     # input_mode='lv1b_plume',  # Switch to LV1B plume-path files from run_lv1_tracking.py.
#     # lv1b_processed_root=Path('/Users/schimmel/data/cosmo-specs/meteograms/cs-eriswil__20260313_111441/lv1_paths'),  # Optional: processed root, cs_run dir, or direct lv1_paths dir.
#     # lv1b_kinds=('extreme', 'integrated', 'vertical'),  # `vertical`: path × level × spectral; altitude collapsed via lv1b_vertical_ice_reduce.
#     # lv1b_tracer='nf',  # Match the tracer segment in `*_plume_path_<tracer>_cell*.nc`.
#     # lv1b_multi_cell_mode='sum',  # Choose `sum`, `mean`, `max`, or `combine` (combine collapses all cells per kind).
#     # lv1b_vertical_ice_reduce='mean',  # For `vertical` kind only: mean, max, sum, p25, p50, p75, p99 along altitude.
#     sw_overrides={
#         'show_psd_twin': True,  # Keep PSD context above each station column.
#         'show_growth_footer': True,  # Ignored in LV1B mode because the moving plume already defines the sample path.
#     },
# )

# runner = SpectralWaterfallNotebook(SETTINGS)
# runner.summary()

In [4]:
# Interactive time: slider updates the figure; no PNG or MP4 files are written.
# Set continuous_slider=True to redraw while dragging (heavier).
# runner.interactive_time(continuous_slider=True)


In [5]:
# mp4_paths[0].resolve().as_uri()

In [6]:
# # Optional batch export (writes PNG frames under output/gfx/png/... and may build MP4):
# mp4_paths = runner.render(mp4=True)
# if mp4_paths:
#     runner.show_mp4(mp4_paths[0])


In [14]:
SETTINGS = SpectralWaterfallSettings(
    # kind='N',
    # input_mode='lv1b_plume',
    # lv1b_processed_root=Path('/Users/schimmel/data/cosmo-specs/meteograms/cs-eriswil__20260313_111441/lv1_paths'),
    # lv1b_kinds=('extreme', 'integrated', 'vertical'),
    # lv1b_vertical_ice_reduce='p75', # 'integrated'
    # lv1b_tracer='nf',
    # lv1b_multi_cell_mode='sum',  # Or: 'mean', 'max', 'combine' (combine collapses all cells per kind).
    # exp_ids=[20260313111448],
    # workers=1,
    # sw_overrides={
    #     'linthresh_W': 1e-2,  # Keep weak liquid-rate structure visible near zero.
    #     'linthresh_F': 1e-2,  # Keep weak ice-rate structure visible near zero.
    #     'xlim_W': [1e-3, 4e3],  # Show the full liquid-diameter range used in the paper.
    #     'xlim_F': [1e-3, 4e3],  # Show the full ice-diameter range used in the paper.
    #     'ylim_W': [-1e6, 1e6],  # Avoid clipping the strongest liquid process pulses.
    #     'ylim_F': [-1e6, 1e6],  # Avoid clipping the strongest ice process pulses.
    #     'psd_ylim_W': [1e1, 1e9],  # Keep liquid PSD visible from first signal to peak.
    #     'psd_ylim_F': [1e1, 1e9],  # Keep ice PSD visible from trace signal to peak
    # },
    kind='Q',
    input_mode='lv1b_plume',
    lv1b_processed_root=Path('/Users/schimmel/data/cosmo-specs/meteograms/cs-eriswil__20260313_111441/lv1_paths'),
    lv1b_kinds=('vertical',),
    lv1b_vertical_ice_reduce='p99', # 'integrated'
    lv1b_tracer='nf',
    lv1b_multi_cell_mode='mean',
    exp_ids=[20260313111448],
    sw_overrides={
        "linthresh_W": 1e-14,  # Preserve subtle liquid mass tendencies around zero.
        "linthresh_F": 1e-14,  # Preserve subtle ice mass tendencies around zero.
        "xlim_W": [1e-3, 4e3],  # Show the full liquid-diameter range used in the paper.
        "xlim_F": [1e-3, 4e3],  # Show the full ice-diameter range used in the paper.
        "ylim_W": [-1e-2, 1e-2],  # Keep liquid mass-rate extremes in frame without flattening onset.
        "ylim_F": [-1e-2, 1e-2],  # Keep ice mass-rate extremes in frame without flattening onset.
        "psd_ylim_W": [1e-8, 1e0],  # Keep liquid mass PSD readable from trace mass to peak.
        "psd_ylim_F": [1e-8, 1e0],  # Keep ice mass PSD readable from trace mass to peak.
    },
)

runner = SpectralWaterfallNotebook(SETTINGS)
runner.summary()

# Optional batch export (writes PNG frames under output/gfx/png/... and may build MP4):
mp4_paths = runner.render(mp4=True)



exp=0 ALLBB: 129 frames
file:///Users/schimmel/Library/Mobile%20Documents/com~apple~CloudDocs/code/polarcap/python/polarcap_analysis/output/gfx/mp4/spectral_waterfall_Q_cs-eriswil__20260313_111441_exp0_stn0-1_ALLBB_evolution_nframes129.mp4
done in 104.7s


In [11]:
if mp4_paths:
    runner.show_mp4(mp4_paths[0])